# Pessoa 2 - Transfer Learning e comparacao final

Este notebook resume a etapa de Transfer Learning do trabalho. Ele nao retreina os modelos; apenas carrega os resultados ja gerados pelos scripts em `src/` e organiza os artefatos para analise, relatorio e apresentacao.

Modelos comparados:

- CNN simples treinada do zero pela Pessoa 1;
- ResNet18 com Transfer Learning;
- DenseNet121 com Transfer Learning, como experimento adicional.

## Configuracao geral

A ResNet18 e a DenseNet121 foram usadas com pesos pre-treinados no ImageNet. Em ambos os casos, o backbone foi congelado e apenas a camada final foi treinada para classificar as imagens em `NORMAL` e `PNEUMONIA`.

A comparacao final usa o conjunto de teste original do dataset. O conjunto de validacao possui apenas 16 imagens, entao suas metricas devem ser interpretadas com cautela.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results"

paths = {
    "simple_cnn_metrics": RESULTS_DIR / "notebook_modelo" / "simple_cnn_metrics.json",
    "resnet_metrics": RESULTS_DIR / "transfer_frozen_modelo" / "transfer_metrics.json",
    "densenet_metrics": RESULTS_DIR / "transfer_densenet121_modelo" / "transfer_metrics.json",
    "comparison_markdown": RESULTS_DIR / "comparison_table.md",
}

missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Arquivos ausentes:\n" + "\n".join(missing))

print(f"Raiz do projeto: {PROJECT_ROOT}")

## Tabela comparativa final

In [ ]:
metric_files = [
    paths["simple_cnn_metrics"],
    paths["resnet_metrics"],
    paths["densenet_metrics"],
]

rows = []
for metric_file in metric_files:
    with metric_file.open("r", encoding="utf-8") as file:
        rows.append(json.load(file))

metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df[["model", "accuracy", "precision", "recall", "f1_score"]]
metrics_df = metrics_df.rename(
    columns={
        "model": "Modelo",
        "accuracy": "Acuracia",
        "precision": "Precisao",
        "recall": "Recall",
        "f1_score": "F1-score",
    }
)

metrics_df.style.format({
    "Acuracia": "{:.4f}",
    "Precisao": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-score": "{:.4f}",
})

## Analise resumida

A ResNet18 com Transfer Learning apresentou o melhor desempenho geral entre os modelos avaliados. Ela superou a CNN simples e a DenseNet121 em acuracia, precisao, recall e F1-score no conjunto de teste.

A DenseNet121 foi mantida como experimento adicional. Apesar de ser uma arquitetura forte e comum em aplicacoes de imagens medicas, neste experimento ela nao superou a ResNet18. Isso reforca a escolha da ResNet18 como modelo final da Pessoa 2.

O recall e uma metrica especialmente relevante neste problema, pois falsos negativos de pneumonia sao indesejaveis. A ResNet18 obteve o maior recall entre os modelos comparados.

## Matrizes de confusao

In [ ]:
confusion_matrices = {
    "CNN simples": RESULTS_DIR / "notebook_modelo" / "simple_cnn_confusion_matrix.png",
    "ResNet18 Transfer Learning": RESULTS_DIR / "transfer_frozen_modelo" / "transfer_confusion_matrix.png",
    "DenseNet121 Transfer Learning": RESULTS_DIR / "transfer_densenet121_modelo" / "transfer_confusion_matrix.png",
}

for title, image_path in confusion_matrices.items():
    if image_path.exists():
        print(title)
        image = mpimg.imread(image_path)
        plt.figure(figsize=(6, 5))
        plt.imshow(image)
        plt.axis("off")
        plt.show()
    else:
        print(f"Arquivo nao encontrado: {image_path}")

## Historico de treinamento dos modelos de Transfer Learning

In [ ]:
training_plots = {
    "ResNet18 - perda": RESULTS_DIR / "transfer_frozen_modelo" / "transfer_loss_history.png",
    "ResNet18 - metricas": RESULTS_DIR / "transfer_frozen_modelo" / "transfer_metrics_history.png",
    "DenseNet121 - perda": RESULTS_DIR / "transfer_densenet121_modelo" / "transfer_loss_history.png",
    "DenseNet121 - metricas": RESULTS_DIR / "transfer_densenet121_modelo" / "transfer_metrics_history.png",
}

for title, image_path in training_plots.items():
    if image_path.exists():
        print(title)
        image = mpimg.imread(image_path)
        plt.figure(figsize=(8, 5))
        plt.imshow(image)
        plt.axis("off")
        plt.show()
    else:
        print(f"Arquivo nao encontrado: {image_path}")

## Comandos usados

Treino da ResNet18 final:

```bash
python -m src.train_transfer --data-dir data/chest_xray --output-dir results/transfer_frozen_modelo --model resnet18 --epochs 10 --batch-size 32 --lr 0.001 --freeze-backbone
```

Experimento adicional com DenseNet121:

```bash
python -m src.train_transfer --data-dir data/chest_xray --output-dir results/transfer_densenet121_modelo --model densenet121 --epochs 10 --batch-size 32 --lr 0.001 --freeze-backbone
```

Tabela comparativa:

```bash
python -m src.compare_results --simple-cnn results/notebook_modelo/simple_cnn_metrics.json --transfer results/transfer_frozen_modelo/transfer_metrics.json --extra-metrics results/transfer_densenet121_modelo/transfer_metrics.json
```

## Conclusao da Pessoa 2

A ResNet18 com Transfer Learning foi escolhida como modelo final da Pessoa 2. Ela apresentou melhor desempenho no teste e melhor equilibrio entre as metricas avaliadas. A DenseNet121 foi testada como alternativa, mas ficou abaixo da ResNet18 neste conjunto de dados.

A principal limitacao metodologica observada foi o tamanho reduzido do conjunto de validacao original. Por isso, a discussao dos resultados deve priorizar as metricas no conjunto de teste, que possui mais imagens e fornece uma avaliacao mais estavel.